In [2]:
import sys, os

# Add the project root (one level up)
PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

print(PROJECT_ROOT)  # sanity check


c:\Users\anaol\repos\langchain\langchain-AI-summit


In [3]:
import re
import uuid
from langsmith.schemas import Run, Example
from langsmith import evaluate, aevaluate, wrappers
from openai import OpenAI
from langsmith import Client
from pydantic import BaseModel
from agent.return_agent import agent
from langchain_core.messages import HumanMessage


# Relevance

In [17]:
def prepare_data(run, example):
    # USER QUESTION (from example input)
    user_msg = example.inputs["messages"][0]["content"]

    # REFERENCE ANSWER (from example output)
    ai_messages = [m for m in example.outputs["messages"] if m["type"] == "ai"]
    reference = ai_messages[-1]["content"] if ai_messages else ""

    # MODEL ANSWER (from run)
    output = run.outputs.get("prediction", "")

    return {
        "question": user_msg,
        "reference": reference,
        "answer": output
    }


In [ ]:
def run_agent(inputs: dict):
    # Extract the user message
    messages = inputs.get("messages", [])
    if not messages:
        raise ValueError("Dataset missing 'messages' list.")

    user_msg = messages[0]["content"]

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Return only final answer
    final_answer = result["messages"][-1].content

    return {"prediction": final_answer}


In [5]:
# Use an LLM-as-a-judge
oai_client = wrappers.wrap_openai(OpenAI())

In [102]:
def relevance(run: Run, example: Example) -> dict:
    """
    Avalia CLAREZA + RELEVÂNCIA da resposta.
    Retorna nota 0..1.
    """
    instructions = """
    Avalie a resposta com base em CLAREZA e RELEVÂNCIA (nota 0 a 1).
    - 1.0 = resposta completa, clara e diretamente relacionada.
    - 0.5 = resposta parcialmente correta.
    - 0.0 = resposta incorreta ou fora do contexto.
    Retorne APENAS a nota, nada mais.
    """

    data = prepare_data(run, example)

    msg = (
        f"Question: {data['question']}\n"
        f"Reference: {data['reference']}\n"
        f"Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    score_str = resp.choices[0].message.content.strip()
    score = float(score_str)

    return {"key": "relevance", "score": score}


In [7]:
client = Client()
client.list_examples(dataset_name="agent-final-output")

<generator object Client.list_examples at 0x000001DE803E6D40>

In [21]:
results = evaluate(
    run_agent,
    data="agent-final-output",
    evaluators=[relevance],
)

View the evaluation results for experiment: 'terrific-son-4' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/cb9bb7e9-467e-48eb-ab7e-1dfa1ff17cac/compare?selectedSessions=1c2f5070-4359-4bb6-baa6-c4947f7c8b97




0it [00:00, ?it/s]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


1it [00:09,  9.71s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


...

5it [01:19, 17.36s/it]

decide_path tool
decision: analyze_seller_reliability
analyze_seller_reliability_node_custom: calling tool directly
Generating final answer...


6it [01:30, 15.46s/it]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


7it [01:44, 14.88s/it]


# Branch Correctness

In [ ]:
from langsmith import Client
ls_client = Client()

examples = [
  {
    "inputs": {"question": "Há quantos pedidos com status cancelado?"},
    "outputs": {"branch": "sql_branch"},
  },
  {
    "inputs": {"question": "Quais 3 vendedores tiveram o maior número de entregas atrasadas em 2024?"},
    "outputs": {"branch": "sql_branch"},
  },
  {
    "inputs": {"question": "No caso de devolução quem arca com custo do envio?"},
    "outputs": {"branch": "pdf_branch"},
  },
  {
    "inputs": {"question": "Qual o prazo de devolução por defeito da BIX?"},
    "outputs": {"branch": "pdf_branch"},
  },
  {
    "inputs": {"question": "O pedido dd787ad9c97e5504d6ea0bd294906902 está dentro do prazo de devolução por arrependimento?"},
    "outputs": {"branch": "pdf_sql_branch"},
  },
  {
    "inputs": {"question": "Quantos pedidos entregues da base de dados estão dentro do prazo de devolução por defeito? Considere que são todos itens não duráveis"},
    "outputs": {"branch": "pdf_sql_branch"},
  },
  {
    "inputs": {"question": "O seller 3442f8959a84dea7ee197c632cb2df15 é confiável?"},
    "outputs": {"branch": "analyze_seller_reliability"},
  },
  {
    "inputs": {"question": "Quais os top 3 vendedores menos confiáveis de 2025?"},
    "outputs": {"branch": "analyze_seller_reliability"},
  },
  {
    "inputs": {"question": "Processe devolução do pedido 2591f6277be80b0c25627c745ec900c4"},
    "outputs": {"branch": "process_return"},
  },
]
dataset = ls_client.create_dataset(dataset_name="branch-decision")
ls_client.create_examples(
  dataset_id=dataset.id,
  examples=examples,
)

In [91]:
def run_agent(inputs: dict):
    # Extract the user message
    user_msg = inputs.get("question", [])

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Return final answer and branch decision
    branch_decision = result.get("decide_path", "unknown")

    return {"branch": branch_decision}


In [ ]:
def branch_correctness(outputs: dict, reference_outputs: dict) -> dict:
    """
    Simple exact match evaluator for branch correctness.
    Checks if the predicted branch exactly matches the expected branch.
    Returns a score of 1.0 if match, 0.0 otherwise.
    """
    predicted_branch = outputs.get("branch", "")
    expected_branch = reference_outputs.get("branch", "")
    
    # Exact match check
    is_correct = predicted_branch == expected_branch
    
    return {
        "key": "branch_correctness",
        "score": 1.0 if is_correct else 0.0
    }


In [95]:
results = evaluate(
    run_agent,
    data="branch-decision",
    evaluators=[branch_correctness],
)

View the evaluation results for experiment: 'healthy-glove-42' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/375cfe16-5c1f-4b73-a81c-1eeb21b6a693/compare?selectedSessions=f3991f91-f671-4f52-9e0c-f6d7d8f274cf




0it [00:00, ?it/s]

decide_path tool
decision: analyze_seller_reliability
analyze_seller_reliability_node_custom: calling tool directly
Generating final answer...


1it [00:14, 14.51s/it]

decide_path tool
decision: pdf_branch
Running PDF branch...
pdf_context loaded: 3710 characters
Generating final answer...


...

7it [01:46, 15.96s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


8it [02:15, 20.02s/it]

decide_path tool
decision: pdf_sql_branch
Running PDF branch...
pdf_context loaded: 3710 characters
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


9it [02:40, 17.85s/it]


# Computation Accuracy

In [ ]:
from langsmith import Client
ls_client = Client()

examples = [
  {
    "inputs": {"question": "Há quantos pedidos com status cancelado?"},
    "outputs": {"numeric_answer": "625"},
  },
  {
    "inputs": {"question": "Quantos pedidos têm método de pagamento cartão de crédito?"},
    "outputs": {"numeric_answer": "76505"},
  },
  {
    "inputs": {"question": "Há quantos clientes com pelo menos um pedido?"},
    "outputs": {"numeric_answer": "99441"},
  },
  {
    "inputs": {"question": "Quantos produtos têm avaliação abaixo de 3 estrelas?"},
    "outputs": {"numeric_answer": "4421"},
  },
  {
    "inputs": {"question": "Quantos pedidos estão em trânsito?"},
    "outputs": {"numeric_answer": "1107"},
  },
]
dataset = ls_client.create_dataset(dataset_name="numeric-answers")
ls_client.create_examples(
  dataset_id=dataset.id,
  examples=examples,
)

{'example_ids': ['63f5f88a-eb92-4caa-9436-67059c49f803',
  '1df398e1-7db3-46c3-ae83-e97948170622',
  'c2e40ed9-f9e6-47d1-8163-8bfc0950e44b',
  'e0e9b879-a6c7-4cae-8d8b-310ac7b0ec84',
  '1b0febfd-1f0b-45c1-9672-005283de5bd7'],
 'count': 5}

In [48]:
def run_agent(inputs: dict):
    # Extract the user message
    user_msg = inputs.get("question", "")

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Extract the final answer from the agent's response
    final_answer = result["messages"][-1].content

    return {"prediction": final_answer}


In [49]:
import re

def extract_first_number(text: str):
    """
    Extracts the first number (int or decimal) from a text.
    Handles '.' or ',' as decimal separators.
    Returns float or None.
    """

    match = re.search(r"\d+[.,]?\d*", text)
    if not match:
        return None

    number_str = match.group(0)

    # Convert comma decimal to dot
    number_str = number_str.replace(",", ".")
    number_str = number_str.replace(".", "")

    return number_str


In [50]:
def computation_accuracy(outputs: dict, reference_outputs: dict) -> dict:
    """
    Simple exact match evaluator for computation accuracy.
    Extracts numeric value from the prediction and compares with expected numeric answer.
    Returns a score of 1.0 if match, 0.0 otherwise.
    """
    import re
    
    # Extract numeric answer from agent's prediction
    prediction = outputs.get("prediction", "")
    
    # This handles cases like "Há 4.421 pedidos" -> extracts "4421"
    predicted_number = extract_first_number(prediction)
    
    # Get expected numeric answer from reference
    expected_number = reference_outputs.get("numeric_answer", "")
    
    # Exact match check (both as strings for exact comparison)
    is_correct = predicted_number == expected_number
    
    return {
        "key": "computation_accuracy",
        "score": 1.0 if is_correct else 0.0
    }


In [51]:
results = evaluate(
    run_agent,
    data="numeric-answers",
    evaluators=[computation_accuracy],
)

View the evaluation results for experiment: 'ample-deer-34' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/1b713cc6-d6cd-482f-9ef3-b3313ee84f95/compare?selectedSessions=20c38498-59e2-4d5a-986b-397a2ce5823a




0it [00:00, ?it/s]

[11/28/25 15:29:28] INFO     Routing request                                                    ]8;id=974943;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=992833;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#584\584]8;;\

[11/28/25 15:29:30] INFO     Fetching database schema                                           ]8;id=562359;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=506909;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#455\455]8;;\

[11/28/25 15:29:31] INFO     Generating SQL query                                               ]8;id=966016;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=169796;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#505\505]8;;\

...

[11/28/25 15:30:29] INFO     Routing request                                                    ]8;id=485060;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=232199;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#584\584]8;;\

[11/28/25 15:30:31] INFO     Fetching database schema                                           ]8;id=814733;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=421773;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#455\455]8;;\

                    INFO     Generating SQL query                                               ]8;id=50441;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=139930;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#505\505]8;;\

[11/28/25 15:31:00] INFO     Generating final answer                                            ]8;id=975698;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=105574;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#520\520]8;;\

5it [01:40, 20.03s/it]


# Relevance + Branch Correctness + Hallucination + Accuracy

In [38]:
def prepare_data(run, example):
    # USER QUESTION (from example input)
    user_msg = example.inputs["messages"][0]["content"]

    # REFERENCE ANSWER (from example output)
    ai_messages = [m for m in example.outputs["messages"] if m["type"] == "ai"]
    reference = ai_messages[-1]["content"] if ai_messages else ""

    # MODEL ANSWER (from run)
    output = run.outputs.get("prediction", "")

    return {
        "question": user_msg,
        "reference": reference,
        "answer": output
    }



In [39]:
# Use an LLM-as-a-judge
oai_client = wrappers.wrap_openai(OpenAI())

In [40]:
bool("True")

True

In [41]:
def extract_binary_score(text: str) -> int:
    """
    Extract the first occurrence of '0' or '1' from a model response.
    Returns 0 if no valid score is found.
    """
    if not text:
        return 0

    # Clean whitespace
    text = text.strip()

    # Look only for a standalone 0 or 1
    for ch in text:
        if ch == "0":
            return 0
        if ch == "1":
            return 1

    return 0


In [42]:
def hallucination(run: Run, example: Example) -> dict:
    """
    Avalia se há HALLUCINAÇÃO na resposta.
    Retorna nota 0 ou 1
    """
    instructions = """
    Você é um avaliador especializado em identificar HALLUCINAÇÕES em respostas de modelos.

    CONTEXTO IMPORTANTE:
    O agente tem acesso a duas fontes de informação confiáveis:
    1. Uma base de dados de e-commerce completa com informações sobre pedidos, clientes, produtos, vendedores, etc.
    2. Um documento PDF com políticas da empresa (políticas de devolução, prazos, regras de frete, etc.)

    INSTRUÇÕES:
    - Resposta com hallucinação é aquela que contem informação que não está no contexto mencionado acima ou está fatualmente errada de acordo com conhecimentos gerais.

    CRITÉRIOS:
    - 1 → Sem halucinação
    - 0 → Resposta com hallucinação

    """

    data = prepare_data(run, example)

    msg = (
        f"Question: {data['question']}\n"
        f"Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    score_str = resp.choices[0].message.content.strip()
    score = extract_binary_score(score_str)

    return {"key": "hallucination", "score": score}


In [43]:
def relevance(run: Run, example: Example) -> dict:
    """
    Avalia CLAREZA + RELEVÂNCIA da resposta.
    Retorna nota 0..1.
    """
    instructions = """
    Avalie a resposta com base em CLAREZA e RELEVÂNCIA (nota 0 a 1).
    - 1.0 = resposta completa, clara e diretamente relacionada.
    - 0.5 = resposta parcialmente correta.
    - 0.0 = resposta incorreta ou fora do contexto.
    Retorne APENAS a nota, nada mais.
    """

    data = prepare_data(run, example)

    msg = (
        f"Question: {data['question']}\n"
        f"Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    score_str = resp.choices[0].message.content.strip()
    score = float(score_str)

    return {"key": "relevance", "score": score}


In [44]:
def accuracy(run: Run, example: Example) -> dict:
    """
    Avalia ACURÁCIA da resposta comparando com a referência.
    Retorna nota 0..1.
    """

    data = prepare_data(run, example)

    instructions = """
    Você é um avaliador especializado em verificar ACURÁCIA de respostas.

    Avalie o quão correta é a resposta do modelo comparada com a RESPOSTA DE REFERÊNCIA.

    CRITÉRIOS (nota de 0.0 a 1.0):
    - 1.0 → A resposta está totalmente correta, sem diferenças factuais relevantes.
    - 0.5 → Parcialmente correta: contém partes verdadeiras, mas erra ou omite detalhes importantes.
    - 0.0 → Incorreta: contradiz a referência ou apresenta informações erradas.

    INSTRUÇÕES:
    - Compare apenas FATO e CONTEÚDO, sem considerar estilo ou redação.
    - Use bom senso para equivalências (“dez” = “10”, sinônimos, ordem invertida, etc.).
    - Retorne APENAS um número entre 0 e 1.
    """

    msg = (
        f"Question: {data['question']}\n"
        f"Reference Answer: {data['reference']}\n"
        f"Model Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    raw = resp.choices[0].message.content.strip()

    # Convert to float safely
    try:
        score = float(raw)
    except:
        import re
        m = re.search(r"\d+(\.\d+)?", raw)
        score = float(m.group(0)) if m else 0.0

    # Clamp 0–1 to prevent model mistakes
    score = max(0.0, min(1.0, score))

    return {"key": "accuracy", "score": score}


In [45]:
# Modified branch_correctness evaluator that works with Run/Example signature
# This allows it to work alongside the relevance evaluator
def branch_correctness(run: Run, example: Example) -> dict:
    """
    Branch correctness evaluator using Run/Example signature.
    Extracts branch from run outputs and compares with example outputs.
    Works with agent-final-output dataset format where decide_path is in the state.
    Returns a score of 1.0 if match, 0.0 otherwise.
    """
    # Extract predicted branch from run outputs
    predicted_branch = run.outputs.get("branch", "")
    
    # Extract expected branch from example outputs
    # For agent-final-output dataset, decide_path is in the output state
    expected_branch = ""
    if example.outputs:
        # Try to get branch from outputs directly (for branch-decision dataset)
        expected_branch = example.outputs.get("branch", "")
        # If not found, try to extract from decide_path in the state (for agent-final-output)
        if not expected_branch:
            expected_branch = example.outputs.get("decide_path", "")
    
    # Exact match check
    is_correct = predicted_branch == expected_branch
    
    return {
        "key": "branch_correctness",
        "score": 1.0 if is_correct else 0.0
    }


In [46]:
# Modified run_agent for agent-final-output dataset that returns both prediction and branch
def run_agent_combined(inputs: dict):
    """
    Modified run_agent that returns both prediction and branch decision.
    Works with agent-final-output dataset format (uses 'messages' input).
    Based on input_example and output_example structure.
    """
    # Extract the user message from inputs (agent-final-output format)
    messages = inputs.get("messages", [])
    if not messages:
        raise ValueError("Dataset missing 'messages' list.")

    user_msg = messages[0]["content"]

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Return both final answer and branch decision
    final_answer = result["messages"][-1].content
    branch_decision = result.get("decide_path", "unknown")

    return {"prediction": final_answer, "branch": branch_decision}


In [47]:
# Run evaluation on agent-final-output dataset with both evaluators
# This evaluates both relevance (LLM-based) and branch correctness (exact match)
results_combined = evaluate(
    run_agent_combined,
    data="agent-final-output",
    evaluators=[relevance, branch_correctness, hallucination, accuracy],
    experiment_prefix="combined-relevance-branch-hallucination",
    description="Combined evaluation of relevance, branch correctness and hallucination on agent-final-output dataset",
)


View the evaluation results for experiment: 'combined-relevance-branch-hallucination-d42751b9' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/cb9bb7e9-467e-48eb-ab7e-1dfa1ff17cac/compare?selectedSessions=8a71b478-c7bb-4b2d-a95e-470fe3e64eab




0it [00:00, ?it/s]

[11/28/25 14:22:04] INFO     Routing request                                                    ]8;id=123773;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=818680;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#584\584]8;;\

[11/28/25 14:22:06] INFO     Processing return order: e481f51cbdc54678b7cc...                   ]8;id=173300;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=205582;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#128\128]8;;\

[11/28/25 14:22:07] INFO     Generating final answer                                            ]8;id=703731;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=587608;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#520\520]8;;\

...

6it [03:08, 31.70s/it]

[11/28/25 14:25:12] INFO     Routing request                                                    ]8;id=144695;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=296469;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#584\584]8;;\

[11/28/25 14:25:16] INFO     Processing return order: e481f51cbdc54678b7cc...                   ]8;id=773902;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=999609;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#128\128]8;;\

                    INFO     Generating final answer                                            ]8;id=96340;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py\return_agent.py]8;;\:]8;id=653918;file://c:\Users\anaol\repos\langchain\langchain-AI-summit\agent\return_agent.py#520\520]8;;\

7it [03:40, 31.54s/it]
